In [1]:
"""
Notebook 06 - Entraînement modèle v2 (PlantVillage + PlantDoc)
AnanthiX AI - Jalon 4
Stratégie : sur-échantillonnage PlantDoc ×10
"""
import os, json, time
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/AnanthiX_AI'

# Vérifier GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
Device : cuda
GPU : NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
PLANTDOC_DIR = '/content/plantdoc'

if not os.path.exists(f'{PLANTDOC_DIR}/train'):
    print("📥 Re-téléchargement PlantDoc...")
    !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !mkdir -p {PLANTDOC_DIR}
    !kaggle datasets download -d nirmalsankalana/plantdoc-dataset -p {PLANTDOC_DIR} --unzip -q
    print(" PlantDoc prêt")
else:
    print(" PlantDoc déjà présent")

📥 Re-téléchargement PlantDoc...
cp: cannot stat '/content/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/plantdoc-dataset
License(s): CC0-1.0
 PlantDoc prêt


In [3]:
# Mapping PlantDoc → tes classes (créé au notebook 05)
with open(f'{PROJECT_ROOT}/results/plantdoc_mapping.json') as f:
    plantdoc_to_my_class = json.load(f)

with open(f'{PROJECT_ROOT}/results/class_names.json') as f:
    class_names = json.load(f)

class_to_idx = {c: i for i, c in enumerate(class_names)}

print(f" {len(class_names)} classes, {len(plantdoc_to_my_class)} mappings PlantDoc")

 38 classes, 28 mappings PlantDoc


In [4]:
import tensorflow_datasets as tfds
import numpy as np
from PIL import Image
from tqdm import tqdm

print(" Chargement PlantVillage depuis tfds (cache)...")
ds, info = tfds.load('plant_village', split='train', as_supervised=True, with_info=True)
tfds_class_names = info.features['label'].names

# Vérifier que l'ordre des classes tfds matche le nôtre
assert tfds_class_names == class_names, " Ordre des classes incohérent !"
print(f" Ordre des classes confirmé")

# Charger les images en RAM (résolution finale 256x256 pour gagner du temps)
IMG_SIZE = 256
pv_images = []
pv_labels = []

print(" Conversion en numpy (peut prendre 3-5 min)...")
for img, label in tqdm(tfds.as_numpy(ds), total=info.splits['train'].num_examples):
    img_pil = Image.fromarray(img).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    pv_images.append(np.array(img_pil, dtype=np.uint8))
    pv_labels.append(int(label))

pv_images = np.stack(pv_images)
pv_labels = np.array(pv_labels)
print(f"\n PlantVillage : {pv_images.shape}, labels {pv_labels.shape}")
print(f"   Taille mémoire : {pv_images.nbytes / 1e9:.2f} GB")

 Chargement PlantVillage depuis tfds (cache)...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/plant_village/incomplete.X2ITY2_1.0.2/plant_village-train.tfrecord-[0-9][0…

Dataset plant_village downloaded and prepared to /root/tensorflow_datasets/plant_village/1.0.2. Subsequent calls will reuse this data.
 Ordre des classes confirmé
 Conversion en numpy (peut prendre 3-5 min)...


100%|██████████| 54303/54303 [00:09<00:00, 5901.36it/s]



 PlantVillage : (54303, 256, 256, 3), labels (54303,)
   Taille mémoire : 10.68 GB


In [5]:
PLANTDOC_TRAIN = f'{PLANTDOC_DIR}/train'
PLANTDOC_TEST = f'{PLANTDOC_DIR}/test'

def load_plantdoc(split_dir):
    images, labels = [], []
    for pd_class, my_class in plantdoc_to_my_class.items():
        class_dir = os.path.join(split_dir, pd_class)
        if not os.path.exists(class_dir):
            continue
        my_idx = class_to_idx[my_class]
        for fname in os.listdir(class_dir):
            try:
                img = Image.open(os.path.join(class_dir, fname)).convert('RGB')
                img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(my_idx)
            except Exception as e:
                print(f" Image ignorée : {fname} ({e})")
    return np.stack(images), np.array(labels)

print(" Chargement PlantDoc train...")
pd_train_images, pd_train_labels = load_plantdoc(PLANTDOC_TRAIN)
print(f" PlantDoc train : {pd_train_images.shape}")

print(" Chargement PlantDoc test (jeu de test terrain)...")
pd_test_images, pd_test_labels = load_plantdoc(PLANTDOC_TEST)
print(f" PlantDoc test : {pd_test_images.shape}")

 Chargement PlantDoc train...
 PlantDoc train : (2670, 256, 256, 3)
 Chargement PlantDoc test (jeu de test terrain)...
 PlantDoc test : (252, 256, 256, 3)


In [6]:
from sklearn.model_selection import train_test_split

# Split 70/15/15 stratifié — MÊMES seed et proportions que la baseline
RANDOM_SEED = 42

idx = np.arange(len(pv_labels))
idx_train, idx_temp, _, y_temp = train_test_split(
    idx, pv_labels, test_size=0.30, random_state=RANDOM_SEED, stratify=pv_labels
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

print(f"PlantVillage train : {len(idx_train)}")
print(f"PlantVillage val   : {len(idx_val)}")
print(f"PlantVillage test  : {len(idx_test)}  (test lab, pour comparer à baseline)")

PlantVillage train : 38012
PlantVillage val   : 8145
PlantVillage test  : 8146  (test lab, pour comparer à baseline)


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

OVERSAMPLE_FACTOR = 10  # PlantDoc répété 10× ← LE PARAMÈTRE CLÉ

# Normalisation ImageNet (identique baseline)
NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

# Augmentation TRAIN (identique baseline)
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.2)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1),
    transforms.ToTensor(),
    NORMALIZE,
])

# Pas d'augmentation pour val/test
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    NORMALIZE,
])

class CombinedDataset(Dataset):
    """Dataset combiné PlantVillage + PlantDoc sur-échantillonné."""
    def __init__(self, pv_imgs, pv_lbls, pd_imgs, pd_lbls, transform, oversample=1):
        self.pv_imgs = pv_imgs
        self.pv_lbls = pv_lbls
        self.pd_imgs = pd_imgs
        self.pd_lbls = pd_lbls
        self.oversample = oversample
        self.transform = transform
        self.n_pv = len(pv_lbls)
        self.n_pd = len(pd_lbls) * oversample if pd_imgs is not None else 0

    def __len__(self):
        return self.n_pv + self.n_pd

    def __getitem__(self, idx):
        if idx < self.n_pv:
            img = self.pv_imgs[idx]
            lbl = self.pv_lbls[idx]
        else:
            # Index dans PlantDoc, avec modulo pour la répétition
            pd_idx = (idx - self.n_pv) % len(self.pd_lbls)
            img = self.pd_imgs[pd_idx]
            lbl = self.pd_lbls[pd_idx]
        return self.transform(img), lbl

# Train : PlantVillage(train) + PlantDoc(train sur-échantillonné ×10)
train_dataset = CombinedDataset(
    pv_imgs=pv_images[idx_train], pv_lbls=pv_labels[idx_train],
    pd_imgs=pd_train_images, pd_lbls=pd_train_labels,
    transform=train_transform, oversample=OVERSAMPLE_FACTOR,
)

# Val : uniquement PlantVillage val (comparaison directe baseline)
val_dataset = CombinedDataset(
    pv_imgs=pv_images[idx_val], pv_lbls=pv_labels[idx_val],
    pd_imgs=None, pd_lbls=None,
    transform=eval_transform, oversample=0,
)

print(f" Composition du training set :")
print(f"   PlantVillage train : {len(idx_train):,} images (×1)")
print(f"   PlantDoc train     : {len(pd_train_labels):,} images (×{OVERSAMPLE_FACTOR})")
print(f"   TOTAL expositions  : {len(train_dataset):,}")
print(f"   Ratio terrain      : {(len(pd_train_labels)*OVERSAMPLE_FACTOR/len(train_dataset))*100:.1f}%")

BATCH_SIZE = 48
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

 Composition du training set :
   PlantVillage train : 38,012 images (×1)
   PlantDoc train     : 2,670 images (×10)
   TOTAL expositions  : 64,712
   Ratio terrain      : 41.3%


In [8]:
import torch.nn as nn
from torchvision import models

NUM_CLASSES = 38

def build_model():
    model = models.resnet50(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

# Partir de la baseline
model = build_model()
baseline_path = f'{PROJECT_ROOT}/models/resnet50_baseline.pth'
state_dict = torch.load(baseline_path, map_location=device)
model.load_state_dict(state_dict)
print(" Baseline chargée comme point de départ")

# Geler Layer1 et Layer2 (identique baseline)
for name, param in model.named_parameters():
    if name.startswith('layer1') or name.startswith('layer2') or name == 'conv1.weight' or name.startswith('bn1'):
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f" Paramètres entraînables : {trainable:,} / {total:,}")

model = model.to(device)

 Baseline chargée comme point de départ
 Paramètres entraînables : 22,140,966 / 23,585,894


In [9]:
import torch.optim as optim

# Config identique baseline pour comparaison juste
LEARNING_RATE = 1e-4
NUM_EPOCHS = 5  # Moins qu'au J2 car on part déjà d'un modèle entraîné

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

print(f"Optimizer : Adam lr={LEARNING_RATE}")
print(f"Epochs : {NUM_EPOCHS}")

Optimizer : Adam lr=0.0001
Epochs : 5


In [10]:
import time
from tqdm import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in tqdm(loader, desc='Train', leave=False):
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc='Val', leave=False):
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
            out = model(imgs)
            loss = criterion(out, lbls)
            total_loss += loss.item() * imgs.size(0)
            correct += (out.argmax(1) == lbls).sum().item()
            total += imgs.size(0)
    return total_loss / total, correct / total

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0
best_state = None
start = time.time()

print(f"\n Entraînement v2 sur {NUM_EPOCHS} epochs\n")
for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  "
          f"train_loss={train_loss:.4f} acc={train_acc:.4f}  |  "
          f"val_loss={val_loss:.4f} acc={val_acc:.4f}  "
          f"({elapsed/60:.1f} min)")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"   Nouveau meilleur modèle (val_acc={val_acc:.4f})")

print(f"\n Entraînement terminé en {(time.time()-start)/60:.1f} min")
print(f"Meilleur val_acc : {best_val_acc:.4f}")

# Charger les meilleurs poids
model.load_state_dict(best_state)


 Entraînement v2 sur 5 epochs



Epoch 1/5  train_loss=0.2128 acc=0.9354  |  val_loss=0.0093 acc=0.9974  (0.8 min)
   Nouveau meilleur modèle (val_acc=0.9974)


Epoch 2/5  train_loss=0.0388 acc=0.9882  |  val_loss=0.0053 acc=0.9984  (0.8 min)
   Nouveau meilleur modèle (val_acc=0.9984)


Epoch 3/5  train_loss=0.0248 acc=0.9922  |  val_loss=0.0081 acc=0.9975  (0.8 min)


Epoch 4/5  train_loss=0.0224 acc=0.9930  |  val_loss=0.0077 acc=0.9980  (0.7 min)


Epoch 5/5  train_loss=0.0092 acc=0.9968  |  val_loss=0.0045 acc=0.9986  (0.8 min)
   Nouveau meilleur modèle (val_acc=0.9986)

 Entraînement terminé en 3.8 min
Meilleur val_acc : 0.9986


<All keys matched successfully>

In [11]:
# Sauvegarder sur Drive
v2_path = f'{PROJECT_ROOT}/models/resnet50_v2.pth'
torch.save(model.state_dict(), v2_path)
print(f" Modèle v2 sauvegardé : {v2_path}")

# Sauvegarder l'historique pour le notebook 07 (comparaison)
history_path = f'{PROJECT_ROOT}/results/training_history_v2.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f" Historique sauvegardé : {history_path}")

# Vérifier les fichiers
import os
size_mb = os.path.getsize(v2_path) / (1024 * 1024)
print(f"\n Taille du modèle v2 : {size_mb:.1f} MB")

 Modèle v2 sauvegardé : /content/drive/MyDrive/AnanthiX_AI/models/resnet50_v2.pth
 Historique sauvegardé : /content/drive/MyDrive/AnanthiX_AI/results/training_history_v2.json

 Taille du modèle v2 : 90.3 MB
